# Phase 7 — chạy xác nhận bản đã sửa (7.15 → 7.14)

Dùng **gói code local đã sửa**, không clone GitHub. Giữ nguyên hai notebook cũ làm hồ sơ. Đây là post-hoc confirmation; evaluation đã được xem trước.

1. Chọn GPU **CUDA tương thích còn đủ VRAM** trong Runtime; không bắt buộc T4. Notebook thử bộ nhớ để chọn batch trước phần chạy dài.
2. Đưa `phase7_corrected_colab_bundle.zip` vào `MyDrive/Fatformer/logs/phase7/`, hoặc tải lên khi được hỏi. Gói không chứa trọng số 2 GB.
3. Checkpoint huấn luyện và ba tar dataset vẫn đọc từ Drive theo cấu trúc đã dùng.
4. Run all. Không sửa T/ngưỡng để ép PASS. G7.3 vẫn pending nếu chưa có reference tốt nhất theo điều kiện.
5. Tải ZIP kết quả ở cell cuối, Save notebook sau chạy. Không ghi đè kết quả cũ.


In [1]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, tarfile, uuid, zipfile

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Fatformer')
assert DRIVE_ROOT.is_dir(), f'Không tìm thấy {DRIVE_ROOT}'
RUN_ID = 'corrected_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
RUN_ROOT = Path('/content/phase7_corrected') / RUN_ID
REPO = RUN_ROOT / 'code'
LOGS = RUN_ROOT / 'logs'
REPO.mkdir(parents=True, exist_ok=False)
LOGS.mkdir()
MAX_BATCH_SIZE = 16  # Giới hạn batch thử; tự giảm khi không đủ VRAM.
print('Run mới:', RUN_ID)


Mounted at /content/drive
Run mới: corrected_20261109_102954_3d28ad


In [2]:
# Nạp snapshot đã sửa. Không GitHub sync, không giải nén phase7_measured_artifacts.zip đè code.
bundle_path = DRIVE_ROOT / 'logs/phase7/phase7_corrected_colab_bundle.zip'
if bundle_path.is_file():
    bundle_bytes = bundle_path.read_bytes()
else:
    print('Chọn đúng file phase7_corrected_colab_bundle.zip từ máy local.')
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Chỉ chọn một ZIP code đã sửa.'
    bundle_bytes = next(iter(uploaded.values()))
input_zip = RUN_ROOT / 'input_bundle.zip'
input_zip.write_bytes(bundle_bytes)
with zipfile.ZipFile(input_zip) as z:
    names = z.namelist()
    assert len(names) == len(set(names)), 'ZIP có tên tệp trùng.'
    manifest = json.loads(z.read('BUNDLE_MANIFEST.json'))
    assert manifest['contract'] == 'phase7-corrected-20261109-v2', 'Sai phiên bản bundle.'
    assert set(names) == set(manifest['files']) | {'BUNDLE_MANIFEST.json'}, 'ZIP thiếu/thừa tệp.'
    for name in names:
        target = (REPO / name).resolve()
        assert target.is_relative_to(REPO.resolve()), f'Đường dẫn ZIP không hợp lệ: {name}'
        data = z.read(name)
        if name != 'BUNDLE_MANIFEST.json':
            assert hashlib.sha256(data).hexdigest() == manifest['files'][name], f'Sai hash: {name}'
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Bundle SHA-256:', hashlib.sha256(bundle_bytes).hexdigest())
print('Nguồn:', manifest['source'])
del bundle_bytes


Bundle SHA-256: 3cd63cd9f5e07d6a70348e3d23809a8216c633b0d45224e815651378923ce657
Nguồn: {'base_git_commit': 'bb43fd90dc65df67707de2eee0c41b5a63eec335', 'working_tree_dirty': True, 'method': 'snapshot of actual working files; not GitHub HEAD'}


In [3]:
# Giữ bộ torch/torchvision có sẵn của Colab; chỉ cài các dependency phụ.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ftfy', 'regex', 'tqdm', 'pyyaml', 'timm', 'einops',
                'scikit-learn', 'pandas', 'matplotlib', 'Pillow'], check=True)
import torch
torch.set_num_threads(4)
os.environ['OMP_NUM_THREADS'] = '4'
torch.manual_seed(42)
assert torch.cuda.is_available(), 'Cần GPU CUDA để chạy benchmark.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU thực tế:', GPU_NAME)
FREE_VRAM, TOTAL_VRAM = torch.cuda.mem_get_info()
print(f'VRAM trống/tổng: {FREE_VRAM/1024**3:.2f}/{TOTAL_VRAM/1024**3:.2f} GiB')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
from tools.phase7_corrected import verify_bundle, validate_inputs, resolve_cfe, resolve_indist, write_json
from tools.calibration_data import sha256_file
bundle_proof = verify_bundle(REPO)
write_json(RUN_ROOT/'environment.json', {'run_id': RUN_ID, 'gpu': GPU_NAME,
    'torch': str(torch.__version__), 'hardware_protocol': 'any_compatible_cuda_gpu',
    'free_vram_gib': FREE_VRAM/1024**3, 'total_vram_gib': TOTAL_VRAM/1024**3,
    'max_batch_size': MAX_BATCH_SIZE, 'bundle': bundle_proof})
print('Đã kiểm tra toàn bộ code/input theo manifest.')

def run_logged(module, arguments):
    log = LOGS/(module.rsplit('.',1)[-1] + '.log')
    command = [sys.executable, '-u', '-m', module] + list(map(str, arguments))
    print('RUN:', ' '.join(command))
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', bufsize=1, cwd=REPO)
        for line in process.stdout:
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        returncode = process.wait()
    assert returncode == 0, f'{module} thất bại. Log: {log}. Không tiếp tục nghiệm thu.'


GPU thực tế: NVIDIA L4
VRAM trống/tổng: 21.84/22.03 GiB
Đã kiểm tra toàn bộ code/input theo manifest.


In [4]:
# Chỉ nhận checkpoint gốc đúng hash, không tự tìm **/best.pth.
REFERENCE_DRIVE = DRIVE_ROOT / 'checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth'
assert REFERENCE_DRIVE.is_file(), f'Thiếu checkpoint: {REFERENCE_DRIVE}'
REFERENCE = RUN_ROOT/'reference/fatformer_anchored_v2_best.pth'
REFERENCE.parent.mkdir()
shutil.copy2(REFERENCE_DRIVE, REFERENCE)
INPUTS = REPO/'inputs'
CALIBRATION = INPUTS/'anchored_v2.json'
CALIB_LOGITS = INPUTS/'calibration_400.npz'
CALIB_SPLIT = INPUTS/'calibration_split.npz'
EVAL_SPLIT = INPUTS/'evaluation_split.npz'
input_proof = validate_inputs(REFERENCE, CALIBRATION, CALIB_LOGITS, CALIB_SPLIT, EVAL_SPLIT)
assert input_proof['partition']['n_calibration'] == 400
assert input_proof['partition']['n_evaluation'] == 1600
assert input_proof['partition']['overlap'] == 0
print('Tiền kiểm calibration:', input_proof)

# Thử bộ nhớ bằng input tổng hợp; không phải số đo accuracy/latency nghiệm thu.
GPU_PROBE = RUN_ROOT/'gpu_memory_probe.json'
run_logged('tools.probe_phase7_gpu', [
    '--checkpoint', REFERENCE, '--output', GPU_PROBE, '--max_batch_size', MAX_BATCH_SIZE])
gpu_probe = json.loads(GPU_PROBE.read_text())
BATCH_SIZE = int(gpu_probe['selected_batch_size'])
assert 1 <= BATCH_SIZE <= MAX_BATCH_SIZE
print('GPU/batch được chọn:', gpu_probe['gpu_name'], BATCH_SIZE)

DATA_ROOT = RUN_ROOT/'data'
CFE_ROOT = DATA_ROOT/'cfe'
INDIST_ROOT = DATA_ROOT/'indist'
archive_hashes = {}
for name, destination in [
    ('test_communityforensics.tar', CFE_ROOT),
    ('test_benchmark_gans.tar', INDIST_ROOT),
    ('test_benchmark_diffusion.tar', INDIST_ROOT),
]:
    archive = DRIVE_ROOT/'datasets'/name
    assert archive.is_file(), f'Thiếu dataset: {archive}'
    archive_hashes[name] = sha256_file(archive)
    destination.mkdir(parents=True, exist_ok=True)
    print('Giải nén:', archive)
    with tarfile.open(archive, 'r:*') as t:
        for member in t.getmembers():
            target = (destination/member.name).resolve()
            assert target.is_relative_to(destination.resolve()), f'Tar path không hợp lệ: {member.name}'
            assert member.isdir() or member.isfile(), f'Không nhận link/device trong dataset: {member.name}'
            assert not (member.isfile() and target.exists()), f'Tệp dataset trùng: {member.name}'
        t.extractall(destination, filter='data')
calib_images, eval_images = resolve_cfe(CFE_ROOT, CALIB_SPLIT, EVAL_SPLIT)
indist_images = resolve_indist(INDIST_ROOT)
assert len(calib_images)==400 and len(eval_images)==1600 and len(indist_images)==82003
write_json(RUN_ROOT/'input_preflight.json', {'inputs': input_proof, 'archives': archive_hashes,
    'n_cfe_calibration': len(calib_images), 'n_cfe_evaluation': len(eval_images),
    'n_indist': len(indist_images)})
print('Dữ liệu đúng: 400 calibration, 1.600 evaluation, 82.003 in-dist.')


Tiền kiểm calibration: {'checkpoint_sha256': '5cd49252ae94189734cee324f9543aa93f462028c879ac259cc16864b609bd3c', 'calibration_sha256': '367c33869d4f41c087f8f5272fd16d2e2ca801515d0dd445302c98a7b254eb65', 'logits_sha256': '3b0f66dfdc7c4655df6aa7fcab7346f0315d2663927559b66005bafa2c657942', 'partition': {'status': 'verified_disjoint_ids', 'identity_unit': 'subset/class/filename; content duplicates not audited', 'n_calibration': 400, 'n_evaluation': 1600, 'overlap': 0, 'calibration_split_sha256': '128176bd4f914ce188be05ef964e05a81802cbf41b140bc99200a4d59646b70c', 'evaluation_split_sha256': 'ed5af91047320147517c7a2a1283e242aaecb0cf9a91351a469902158b8bc4e9', 'sample_ids_sha256': '5ae613da819d065b93225b901a7d06d3309ad27e8cdbb9a441cf4df43d7552fb'}, 'fpr_calibration': 0.05, 'T': 2.7863595485687256, 'threshold': 0.7953211069107056, 'inference_precision': 'fp32', 'extraction_batch_size': 32, 'probability_device': 'cpu', 'probability_batch_size': 32}
RUN: /usr/bin/python3 -u -m tools.probe_phase7_g

In [5]:
# 10 real + 10 fake từ calibration, chỉ dùng kiểm tra hành vi đóng gói.
VERIFY_IMAGES = RUN_ROOT/'verification_images'
VERIFY_IMAGES.mkdir()
selected = []
for subset in sorted({x[2].split('/')[0] for x in calib_images}):
    for label in [0, 1]:
        chosen = [x for x in calib_images if x[1]==label and x[2].split('/')[0]==subset][:2]
        assert len(chosen)==2, f'Thiếu ảnh verification: {subset}/{label}'
        selected.extend(chosen)
assert len(selected)==20
for index,(path,label,sid) in enumerate(selected):
    shutil.copy2(path, VERIFY_IMAGES/(f'{index:02d}_' + Path(path).name))
write_json(RUN_ROOT/'verification_image_ids.json',
           [{'sample_id': sid, 'target': label} for path,label,sid in selected])


In [6]:
# Đo lại đúng 400 ảnh, cùng FP16/batch đã chọn với benchmark.
# Bước ngắn này loại bỏ sai khác precision giữa cache FP32 cũ và triển khai FP16.
CALIB_RUNTIME = RUN_ROOT/'calibration'
CALIB_RUNTIME.mkdir()
CALIB_LOGITS = CALIB_RUNTIME/'logits_anchored_v2_calib_fp16.npz'
CALIBRATION = CALIB_RUNTIME/'anchored_v2.json'
run_logged('tools.extract_calibration_logits', [
    '--checkpoint', REFERENCE, '--data_dir', CFE_ROOT,
    '--split_file', CALIB_SPLIT, '--evaluation_split', EVAL_SPLIT,
    '--output_npz', CALIB_LOGITS, '--device', 'cuda', '--precision', 'fp16', '--batch_size', BATCH_SIZE])
run_logged('tools.calibrate_and_select', [
    '--checkpoint', REFERENCE, '--logits_npz', CALIB_LOGITS,
    '--calibration_split', CALIB_SPLIT, '--evaluation_split', EVAL_SPLIT,
    '--model_name', 'anchored_v2', '--output_dir', CALIB_RUNTIME, '--fpr_target', '0.05',
    '--probability_device', 'cuda'])
runtime_calibration = validate_inputs(REFERENCE, CALIBRATION, CALIB_LOGITS, CALIB_SPLIT, EVAL_SPLIT)
assert runtime_calibration['inference_precision']=='fp16'
assert runtime_calibration['extraction_batch_size']==BATCH_SIZE
write_json(RUN_ROOT/'runtime_calibration_preflight.json', runtime_calibration)
print('Calibration FP16 trên đúng 400 ảnh:', runtime_calibration)


RUN: /usr/bin/python3 -u -m tools.extract_calibration_logits --checkpoint /content/phase7_corrected/corrected_20261109_102954_3d28ad/reference/fatformer_anchored_v2_best.pth --data_dir /content/phase7_corrected/corrected_20261109_102954_3d28ad/data/cfe --split_file /content/phase7_corrected/corrected_20261109_102954_3d28ad/code/inputs/calibration_split.npz --evaluation_split /content/phase7_corrected/corrected_20261109_102954_3d28ad/code/inputs/evaluation_split.npz --output_npz /content/phase7_corrected/corrected_20261109_102954_3d28ad/calibration/logits_anchored_v2_calib_fp16.npz --device cuda --precision fp16 --batch_size 16
[*] Tìm thấy 400 mẫu hiệu chuẩn (Real: 200, Fake: 200).
[*] Đang nạp checkpoint: /content/phase7_corrected/corrected_20261109_102954_3d28ad/reference/fatformer_anchored_v2_best.pth
[✓] Nạp mô hình thành công. Đang tiến hành forward trích xuất logits...
[✓] Đã xuất 400 logits đo thật ra tệp NPZ: /content/phase7_corrected/corrected_20261109_102954_3d28ad/calibratio

In [7]:
# Đóng gói trực tiếp từ checkpoint huấn luyện trên Drive + calibration 400 ảnh đã kiểm tra.
PACKAGE_DIR = RUN_ROOT/'package'
PACKAGE = PACKAGE_DIR/'unified_fatformer_v2.pth'
PACKAGE_DIR.mkdir()
run_logged('tools.package_unified_fatformer', [
    '--checkpoint', REFERENCE, '--calibration', CALIBRATION,
    '--output', PACKAGE, '--metadata_out', PACKAGE_DIR/'metadata.json'])
PACKAGE_SHA = sha256_file(PACKAGE)
print('Package mới SHA-256:', PACKAGE_SHA)


RUN: /usr/bin/python3 -u -m tools.package_unified_fatformer --checkpoint /content/phase7_corrected/corrected_20261109_102954_3d28ad/reference/fatformer_anchored_v2_best.pth --calibration /content/phase7_corrected/corrected_20261109_102954_3d28ad/calibration/anchored_v2.json --output /content/phase7_corrected/corrected_20261109_102954_3d28ad/package/unified_fatformer_v2.pth --metadata_out /content/phase7_corrected/corrected_20261109_102954_3d28ad/package/metadata.json
[*] Đang đọc checkpoint nguồn: /content/phase7_corrected/corrected_20261109_102954_3d28ad/reference/fatformer_anchored_v2_best.pth
[✓] Checkpoint SHA-256: 5cd49252ae94189734cee324f9543aa93f462028c879ac259cc16864b609bd3c
[✓] Layout adapter nhận diện: n=3, indices=[7, 15, 23], partial=False
[*] Đang nạp calibration: /content/phase7_corrected/corrected_20261109_102954_3d28ad/calibration/anchored_v2.json
[✓] Xác thực calibration chính chủ: Khớp SHA-256 100%.
[*] Đang lưu checkpoint đóng gói ra: /content/phase7_corrected/correc

In [8]:
VERIFICATION = RUN_ROOT/'real_image_verification.json'
run_logged('tools.verify_unified_package', [
    '--checkpoint', PACKAGE, '--reference', REFERENCE, '--output', VERIFICATION,
    '--device', 'cuda', '--precision', 'fp16', '--images', VERIFY_IMAGES])
proof = json.loads(VERIFICATION.read_text())
assert proof['checkpoint_sha256'] == PACKAGE_SHA
assert proof['real_image_acceptance'] is True and proof['n_images'] == 20
print('20 ảnh + toàn bộ tensor + standalone: PASS. Đây chưa phải đối đầu với best model theo điều kiện.')


RUN: /usr/bin/python3 -u -m tools.verify_unified_package --checkpoint /content/phase7_corrected/corrected_20261109_102954_3d28ad/package/unified_fatformer_v2.pth --reference /content/phase7_corrected/corrected_20261109_102954_3d28ad/reference/fatformer_anchored_v2_best.pth --output /content/phase7_corrected/corrected_20261109_102954_3d28ad/real_image_verification.json --device cuda --precision fp16 --images /content/phase7_corrected/corrected_20261109_102954_3d28ad/verification_images
/content/phase7_corrected/corrected_20261109_102954_3d28ad/code/src/models/unified_fatformer.py:165: RuntimeWarning: [WARN] router disabled: no validated held-out win; calibrated fixed fallback
  warnings.warn("[WARN] router disabled: no validated held-out win; calibrated fixed fallback", RuntimeWarning)
{
  "schema": "unified_package_verification/v1",
  "checkpoint_sha256": "248690528f6e7ecadb06717fafbdb64ca5c101cb966ad3f06f8c00994c9245d0",
  "reference_sha256": "5cd49252ae94189734cee324f9543aa93f462028c

In [9]:
# Chấm thực tế qua package.predict; giữ toàn bộ logits/prob/nhãn/ID, không dùng cache.
BENCHMARK = RUN_ROOT/'benchmark'
run_logged('tools.run_phase7_corrected', [
    '--bundle_root', REPO, '--package', PACKAGE, '--reference_checkpoint', REFERENCE,
    '--calibration', CALIBRATION, '--calibration_logits', CALIB_LOGITS,
    '--calibration_split', CALIB_SPLIT, '--evaluation_split', EVAL_SPLIT,
    '--cfe_root', CFE_ROOT, '--indist_root', INDIST_ROOT,
    '--verification', VERIFICATION, '--output_dir', BENCHMARK,
    '--device', 'cuda', '--precision', 'fp16', '--batch_size', BATCH_SIZE, '--num_workers', '2'])
result = json.loads((BENCHMARK/'final_report.json').read_text())
assert result['status'] == 'completed'
print('G7.2:', result['gates']['G7.2'])


RUN: /usr/bin/python3 -u -m tools.run_phase7_corrected --bundle_root /content/phase7_corrected/corrected_20261109_102954_3d28ad/code --package /content/phase7_corrected/corrected_20261109_102954_3d28ad/package/unified_fatformer_v2.pth --reference_checkpoint /content/phase7_corrected/corrected_20261109_102954_3d28ad/reference/fatformer_anchored_v2_best.pth --calibration /content/phase7_corrected/corrected_20261109_102954_3d28ad/calibration/anchored_v2.json --calibration_logits /content/phase7_corrected/corrected_20261109_102954_3d28ad/calibration/logits_anchored_v2_calib_fp16.npz --calibration_split /content/phase7_corrected/corrected_20261109_102954_3d28ad/code/inputs/calibration_split.npz --evaluation_split /content/phase7_corrected/corrected_20261109_102954_3d28ad/code/inputs/evaluation_split.npz --cfe_root /content/phase7_corrected/corrected_20261109_102954_3d28ad/data/cfe --indist_root /content/phase7_corrected/corrected_20261109_102954_3d28ad/data/indist --verification /content/ph

In [10]:
LATENCY_BATCHES = sorted({1, BATCH_SIZE, 8, 32})
LATENCY = RUN_ROOT/'latency_phase7.json'
run_logged('tools.infer_unified', [
    '--checkpoint', PACKAGE, '--device', 'cuda',
    '--benchmark_latency', '--latency_out', LATENCY,
    '--latency_batches', *map(str, LATENCY_BATCHES)])
run_logged('tools.finalize_phase7_corrected', [
    '--benchmark', BENCHMARK/'final_report.json', '--verification', VERIFICATION,
    '--latency', LATENCY, '--output', RUN_ROOT/'acceptance.json'])
print((RUN_ROOT/'acceptance.json').read_text())
print('Không đổi FAIL/pending thành PASS. Chưa có best-reference theo điều kiện thì G7.3 vẫn pending.')


RUN: /usr/bin/python3 -u -m tools.infer_unified --checkpoint /content/phase7_corrected/corrected_20261109_102954_3d28ad/package/unified_fatformer_v2.pth --device cuda --benchmark_latency --latency_out /content/phase7_corrected/corrected_20261109_102954_3d28ad/latency_phase7.json --latency_batches 1 8 16 32
[*] Khởi tạo môi trường suy luận trên thiết bị: cuda
/content/phase7_corrected/corrected_20261109_102954_3d28ad/code/src/models/unified_fatformer.py:165: RuntimeWarning: [WARN] router disabled: no validated held-out win; calibrated fixed fallback
  warnings.warn("[WARN] router disabled: no validated held-out win; calibrated fixed fallback", RuntimeWarning)
[✓] Nạp UnifiedFatFormerXLA thành công!

TIẾN HÀNH ĐO ĐẠC ĐỘ TRỄ SUY LUẬN (LATENCY OVERHEAD BENCHMARK - G7.3)
Thiết bị: cuda | Precision: FP16 | Warmup: 20 | Steps: 200
• Batch  1: Package =  44.98 ms/batch | 44.98 ms/ảnh | Overhead = 0.1881 ms |  22.2 FPS
• Batch  8: Package =  76.79 ms/batch |  9.60 ms/ảnh | Overhead = 0.0596 ms 

In [11]:
# Có thể chạy cell này riêng sau lỗi để lấy log. Không ghi đè hồ sơ/weights của run cũ.
DRIVE_RUN = DRIVE_ROOT/'logs/phase7'/RUN_ID
DRIVE_MODEL = DRIVE_ROOT/'checkpoint/phase7'/RUN_ID
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
DRIVE_MODEL.mkdir(parents=True, exist_ok=False)
if (RUN_ROOT/'package/unified_fatformer_v2.pth').is_file():
    shutil.copy2(RUN_ROOT/'package/unified_fatformer_v2.pth', DRIVE_MODEL/'unified_fatformer_v2.pth')
    if (RUN_ROOT/'package/metadata.json').is_file():
        shutil.copy2(RUN_ROOT/'package/metadata.json', DRIVE_MODEL/'metadata.json')
result_zip = Path('/content')/(RUN_ID + '_results.zip')
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN_ROOT.rglob('*'):
        if not path.is_file() or path.is_relative_to(RUN_ROOT/'data') or path.is_relative_to(REPO):
            continue
        if path.suffix.lower() in {'.pth','.pt'}:
            continue
        z.write(path, path.relative_to(RUN_ROOT).as_posix())
    z.write(REPO/'BUNDLE_MANIFEST.json', 'BUNDLE_MANIFEST.json')
shutil.copy2(result_zip, DRIVE_RUN/result_zip.name)
print('ZIP kết quả (không kèm weights 2 GB):', DRIVE_RUN/result_zip.name)
print('Weights theo run:', DRIVE_MODEL/'unified_fatformer_v2.pth')
print('Save notebook đã chạy, tải ZIP này về folder wait để hậu kiểm.')
files.download(str(result_zip))


ZIP kết quả (không kèm weights 2 GB): /content/drive/MyDrive/Fatformer/logs/phase7/corrected_20261109_102954_3d28ad/corrected_20261109_102954_3d28ad_results.zip
Weights theo run: /content/drive/MyDrive/Fatformer/checkpoint/phase7/corrected_20261109_102954_3d28ad/unified_fatformer_v2.pth
Save notebook đã chạy, tải ZIP này về folder wait để hậu kiểm.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>